# StaR — Stability Benchmark
Reproduce the 500-seed ARI variance results from the proposal.

In [ ]:
import sys; sys.path.insert(0, '..')
from star.data import make_synthetic_dlpfc
from star.model import NumpyGraphAutoEncoder
from star.wrapper import NumpyStaRSimulator
from star.benchmark import StabilityBenchmark
from star.visualize import plot_violin, plot_ablation_table

ds    = make_synthetic_dlpfc(n_spots=3000, n_genes=3000, n_layers=7, seed=0)
bench = StabilityBenchmark(n_seeds=100, n_clusters=7)

In [ ]:
# Baseline factory
def make_baseline(seed):
    import numpy as np; np.random.seed(seed)
    m = NumpyGraphAutoEncoder(ds.n_genes, [512, 30], seed=seed)
    m.fit(ds.X, ds.edge_index, n_epochs=60)
    return m

# StaR factory
def make_star(seed):
    import numpy as np; np.random.seed(seed)
    base = NumpyGraphAutoEncoder(ds.n_genes, [512, 30], seed=seed)
    star = NumpyStaRSimulator(base, lambda1=0.5, lambda2=0.3, lambda3=0.1)
    star.fit(ds.X, ds.edge_index, n_epochs=60)
    return star

print('Running baseline (100 seeds)...')
res_base = bench.run(make_baseline, ds, label='STAGATE (baseline)')
print('Running StaR (100 seeds)...')
res_star = bench.run(make_star,     ds, label='StaR (all three)')
bench.print_table([res_base, res_star])

In [ ]:
plot_violin([res_base, res_star], 'violin_nb.png')
from IPython.display import Image; Image('violin_nb.png')